<a href="https://colab.research.google.com/github/tony-tokenomics-net/craftguide/blob/main/NexBridge_TIA_Dataset_Generator_v6_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NexBridge TIA Dataset Generator
**Trajectory Intelligence Assessment — Synthetic Organizational Dataset**

This notebook generates a realistic synthetic TIA survey dataset for **NexBridge**, a PE-backed
tech-enabled workforce compliance and benefits administration company (~500 employees).

The dataset is constructed **business unit by business unit**, then merged into a single
organization-level file. Each BU section has clearly labeled input parameters so this notebook
can be reused as a **client-configurable module** — adjust the parameters for any engagement.

**Output:** A single CSV with one row per respondent containing:
- Demographic fields (BU, job level, employee ID)
- 61 raw survey items (Q1–Q61) in corporate terminology, reverse-scored items left as-is
  (downstream scoring models handle the reversal)
- Computed subscale means for QA inspection

**Instrument version:** TIA V4.0 — 61 items, 1–6 Likert scale


## 0. Imports & Global Configuration

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
# GLOBAL RANDOM SEED
# Change this to produce a different but reproducible dataset.
# ─────────────────────────────────────────────────────────────
MASTER_SEED = 2024

rng = np.random.default_rng(MASTER_SEED)


## 1. Corporate Terminology Map
Academic → Corporate conversion per the TIA Terminology Conversion Guide.
Column names in the output dataset use corporate terminology.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TERMINOLOGY MAP  (Academic → Corporate)
# Source: Trajectory Intelligence - Terminology Conversion Guide
# Do NOT change these — they control output column naming throughout.
# ─────────────────────────────────────────────────────────────────────────────
TERM = {
    # Demands
    "D1": "decision_density",          # Cognitive Load
    "D2": "workload_compression",      # Time Pressure
    "D3": "interpersonal_complexity",  # Client & Service Complexity
    "D4": "environmental_interference",# Physical Work Environment
    "D5": "boundary_permeability",     # Schedule & Boundary Demands
    # Resources
    "R1": "feedback_clarity",          # Feedback
    "R2": "rewards_recognition",       # Rewards & Recognition (kept original)
    "R3": "decision_latitude",         # Job Control & Autonomy
    "R4": "influence_inclusion",       # Participation & Voice
    "R5": "job_security",              # Job Security (kept original)
    "R6": "leadership_reliability",    # Supervisor Support
    # 4R
    "REC": "recognize",
    "RES": "respond",
    "REV": "resolve",
    "REF": "refine",
    # Outcomes
    "EXH": "execution_drag",           # Exhaustion
    "DIS": "commitment_drift",         # Disengagement
}

print("Terminology map loaded.")
print({k: v for k, v in TERM.items()})


Terminology map loaded.
{'D1': 'decision_density', 'D2': 'workload_compression', 'D3': 'interpersonal_complexity', 'D4': 'environmental_interference', 'D5': 'boundary_permeability', 'R1': 'feedback_clarity', 'R2': 'rewards_recognition', 'R3': 'decision_latitude', 'R4': 'influence_inclusion', 'R5': 'job_security', 'R6': 'leadership_reliability', 'REC': 'recognize', 'RES': 'respond', 'REV': 'resolve', 'REF': 'refine', 'EXH': 'execution_drag', 'DIS': 'commitment_drift'}


## 2. Reverse Scoring Registry
All 61 items with their reverse-scoring flag, per TIA V4.0 (61-Question Set).

**Important:** Raw item responses are stored AS-IS (including reversed items).
The downstream scoring models handle reversal. The map here is for documentation
and QA only — it does not transform any values during dataset generation.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# REVERSE SCORING MAP — TIA V4.0, 61 Items
# True = item is reverse-scored (higher raw = LESS of the construct)
# Source: TIA_-_61-Question_Set__V4_0_.xlsx
# ─────────────────────────────────────────────────────────────────────────────
REVERSE_MAP = {
    # ── Demands ──────────────────────────────────────────────────────────────
    # D1 – Decision Density
    "D1Q1": False,   # forward
    "D1Q2": True,    # reversed
    "D1Q3": True,    # reversed

    # D2 – Workload Compression
    "D2Q1": False,
    "D2Q2": True,    # reversed
    "D2Q3": False,

    # D3 – Interpersonal Complexity
    "D3Q1": False,
    "D3Q2": True,    # reversed
    "D3Q3": False,

    # D4 – Environmental Interference
    "D4Q1": True,    # reversed
    "D4Q2": True,    # reversed
    "D4Q3": False,

    # D5 – Boundary Permeability
    "D5Q1": True,    # reversed
    "D5Q2": False,
    "D5Q3": False,

    # ── Resources ────────────────────────────────────────────────────────────
    # R1 – Feedback Clarity
    "R1Q1": True,    # reversed
    "R1Q2": False,
    "R1Q3": True,    # reversed

    # R2 – Rewards & Recognition
    "R2Q1": True,    # reversed
    "R2Q2": False,
    "R2Q3": True,    # reversed

    # R3 – Decision Latitude
    "R3Q1": False,
    "R3Q2": False,
    "R3Q3": True,    # reversed

    # R4 – Influence & Inclusion
    "R4Q1": False,
    "R4Q2": True,    # reversed
    "R4Q3": True,    # reversed

    # R5 – Job Security
    "R5Q1": False,
    "R5Q2": True,    # reversed
    "R5Q3": False,

    # R6 – Leadership Reliability
    "R6Q1": False,
    "R6Q2": True,    # reversed
    "R6Q3": False,

    # ── 4R Capability ─────────────────────────────────────────────────────────
    # Recognize
    "RECQ1": True,   # reversed
    "RECQ2": True,   # reversed
    "RECQ3": False,

    # Respond
    "RESQ1": True,   # reversed
    "RESQ2": False,
    "RESQ3": False,

    # Resolve
    "REVQ1": True,   # reversed
    "REVQ2": False,
    "REVQ3": True,   # reversed

    # Refine
    "REFQ1": True,   # reversed
    "REFQ2": False,
    "REFQ3": True,   # reversed

    # ── Execution Drag ────────────────────────────────────────────────────────
    "EXHQ1": False,
    "EXHQ2": False,
    "EXHQ3": True,   # reversed
    "EXHQ4": False,
    "EXHQ5": True,   # reversed
    "EXHQ6": True,   # reversed
    "EXHQ7": False,
    "EXHQ8": True,   # reversed

    # ── Commitment Drift ──────────────────────────────────────────────────────
    "DISQ1": False,
    "DISQ2": False,
    "DISQ3": True,   # reversed
    "DISQ4": False,
    "DISQ5": True,   # reversed
    "DISQ6": True,   # reversed
    "DISQ7": False,
    "DISQ8": True,   # reversed
}

reverse_items = [k for k, v in REVERSE_MAP.items() if v]
forward_items = [k for k, v in REVERSE_MAP.items() if not v]
print(f"Total items: {len(REVERSE_MAP)}  |  Reversed: {len(reverse_items)}  |  Forward: {len(forward_items)}")
print(f"Reversed items: {reverse_items}")


Total items: 61  |  Reversed: 31  |  Forward: 30
Reversed items: ['D1Q2', 'D1Q3', 'D2Q2', 'D3Q2', 'D4Q1', 'D4Q2', 'D5Q1', 'R1Q1', 'R1Q3', 'R2Q1', 'R2Q3', 'R3Q3', 'R4Q2', 'R4Q3', 'R5Q2', 'R6Q2', 'RECQ1', 'RECQ2', 'RESQ1', 'REVQ1', 'REVQ3', 'REFQ1', 'REFQ3', 'EXHQ3', 'EXHQ5', 'EXHQ6', 'EXHQ8', 'DISQ3', 'DISQ5', 'DISQ6', 'DISQ8']


## 3. Core Generation Engine
Shared helper functions used by every BU generator.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def z_score(x):
    """Standardize a 1-D array."""
    x = np.asarray(x, dtype=float)
    return (x - x.mean()) / (x.std(ddof=0) + 1e-9)


def to_likert_6(latent, intercept=3.5, loading=0.75, error_sd=0.60, rng=rng):
    """Map a continuous latent variable to a 1–6 Likert response."""
    raw = intercept + loading * np.asarray(latent, dtype=float) + rng.normal(0, error_sd, len(latent))
    return np.clip(np.rint(raw), 1, 6).astype(int)


def reverse_6(x):
    """Reverse a 1–6 Likert item: 1↔6, 2↔5, 3↔4."""
    return (7 - np.asarray(x, dtype=int))


def intercept_for_target(target_mean, scale_min=1, scale_max=6):
    """
    Return an approximate intercept that, with loading=0.75 and a zero-mean
    latent, produces responses centred near target_mean on a 1–6 scale.
    Clipped to keep outputs in [scale_min, scale_max].
    """
    return float(np.clip(target_mean, scale_min + 0.5, scale_max - 0.5))


def build_latent(n, mean_shift=0.0, level_effect=-0.05, z_level=None, rng=rng, noise=0.85):
    """
    Build a single standardised latent construct.

    Parameters
    ----------
    n            : number of respondents
    mean_shift   : constant push (+high / -low) — tune target subscale mean
    level_effect : how much job level modifies the construct (usually small)
    z_level      : pre-computed standardised job_level array (length n)
    noise        : std of individual error
    """
    base = mean_shift + rng.normal(0, noise, n)
    if z_level is not None:
        base = base + level_effect * z_level
    return base


def make_items_for_dim(latent, item_keys, reverse_map, intercept=3.5,
                       loading=0.75, error_sd=0.60, rng=rng):
    """
    Generate Likert-6 item responses for a single subscale dimension.

    For reversed items the RAW score is stored (higher = less of construct),
    matching what an actual survey platform would export. The reverse_map
    controls which items receive this treatment.

    Parameters
    ----------
    latent      : latent construct array (n,)
    item_keys   : list of Q-column names, e.g. ['Q1','Q2','Q3']
    reverse_map : the global REVERSE_MAP dict
    intercept   : scale midpoint for to_likert_6
    """
    out = {}
    for key in item_keys:
        raw = to_likert_6(latent, intercept=intercept, loading=loading,
                          error_sd=error_sd, rng=rng)
        if reverse_map[key]:
            # Store the REVERSED response as a real survey would deliver it
            out[key] = reverse_6(raw)
        else:
            out[key] = raw
    return out


# ─────────────────────────────────────────────────────────────────────────────
# SUBSCALE QA HELPER
# Computes "effective" (scored-direction) means for QA only.
# This is NOT stored in the output — the raw items are stored.
# ─────────────────────────────────────────────────────────────────────────────
def subscale_mean_qa(df_items, item_keys, reverse_map):
    """Return mean of items after applying reversal for QA inspection."""
    scored = []
    for k in item_keys:
        col = df_items[k].copy()
        if reverse_map[k]:
            col = reverse_6(col)
        scored.append(col)
    return pd.concat(scored, axis=1).mean(axis=1).mean()


print("Core generation engine loaded.")


Core generation engine loaded.


## 4. NexBridge Company Configuration
**Primary adjustment panel.** Change values here to reconfigure for a different client.


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# ■ NEXBRIDGE — COMPANY-LEVEL CONFIGURATION
# ═════════════════════════════════════════════════════════════════════════════

COMPANY_NAME   = "NexBridge"
INDUSTRY       = "Tech-Enabled Workforce Compliance & Benefits Administration"
REVENUE_ARR    = "~$85–110M"          # Informational only; not in dataset
ACQUISITION    = "18 months ago"       # Informational only
BOLT_ON        = "6 months post-close" # Informational only

# ── Org Size & Participation ──────────────────────────────────────────────────
TOTAL_EMPLOYEES      = 401
PARTICIPATION_RATE   = 0.79            # 79%  →  ~395 respondents

# ── Job Levels ────────────────────────────────────────────────────────────────
# 1 = Top Leadership  …  6 = Associate / Individual Contributor
JOB_LEVELS = [1, 2, 3, 4, 5, 6]

# ── Participation Distribution ────────────────────────────────────────────────
# Participation is concentrated in levels 3–6 (front-line).
# Leadership (levels 1–2) participates at a higher rate — common in real surveys.
LEVEL_PARTICIPATION_WEIGHTS = {
    1: 1.00,   # C-suite/VP — high participation
    2: 0.85,   # Director/Senior Manager
    3: 0.82,   # Manager
    4: 0.78,   # Senior IC
    5: 0.65,   # IC
    6: 0.61,   # Associate
}

# ── Business Unit Headcounts (Total Employees) ───────────────────────────────
BU_HEADCOUNTS = {
    "Sales":            146,
    "Customer_Success":  81,
    "Operations":        74,
    "Product_Software":  31,
    "Marketing":         28,
    "IT":                19,
    "Finance":            9,
    "HR":                 7,
    "Leadership":         6,
}
assert sum(BU_HEADCOUNTS.values()) == TOTAL_EMPLOYEES, \
    f"BU headcounts sum to {sum(BU_HEADCOUNTS.values())}, expected {TOTAL_EMPLOYEES}"

# ── Calibrate level weights to hit org-wide participation target ──────────────
# LEVEL_PARTICIPATION_WEIGHTS defines the *shape* (relative likelihood by level).
# The calibration step scales them so their weighted average equals PARTICIPATION_RATE.

_raw_weights = np.array([LEVEL_PARTICIPATION_WEIGHTS[lv] for lv in JOB_LEVELS])
_scale_factor = PARTICIPATION_RATE / _raw_weights.mean()
CALIBRATED_WEIGHTS = {
    lv: min(LEVEL_PARTICIPATION_WEIGHTS[lv] * _scale_factor, 1.0)
    for lv in JOB_LEVELS
}

print("Calibrated participation weights:")
for lv, w in CALIBRATED_WEIGHTS.items():
    print(f"  Level {lv}: {w:.3f}")
print(f"  Weighted avg: {np.mean(list(CALIBRATED_WEIGHTS.values())):.3f}  (target: {PARTICIPATION_RATE})")
print(f"Company : {COMPANY_NAME}")
print(f"Industry: {INDUSTRY}")
print(f"Revenue : {REVENUE_ARR} ARR")
print(f"\nTotal employees : {TOTAL_EMPLOYEES}")
print(f"Participation   : {PARTICIPATION_RATE:.0%}")
print(f"Expected n      : ~{int(TOTAL_EMPLOYEES * PARTICIPATION_RATE)}")
print(f"\nBU headcounts:")
for bu, n in BU_HEADCOUNTS.items():
    print(f"  {bu:<22} {n:>3} employees")


Calibrated participation weights:
  Level 1: 1.000
  Level 2: 0.855
  Level 3: 0.825
  Level 4: 0.785
  Level 5: 0.654
  Level 6: 0.614
  Weighted avg: 0.789  (target: 0.79)
Company : NexBridge
Industry: Tech-Enabled Workforce Compliance & Benefits Administration
Revenue : ~$85–110M ARR

Total employees : 401
Participation   : 79%
Expected n      : ~316

BU headcounts:
  Sales                  146 employees
  Customer_Success        81 employees
  Operations              74 employees
  Product_Software        31 employees
  Marketing               28 employees
  IT                      19 employees
  Finance                  9 employees
  HR                       7 employees
  Leadership               6 employees


## 5. Business Unit Score Profiles
**Subscale target means on a 1–6 scale.** These are the primary levers for
shaping the narrative each BU tells. Higher = more of the construct.

For **Demand** items: higher = more demand (worse).
For **Resource** items: higher = more resource (better).
For **4R Capability**: higher = more mature capability (better).
For **Execution Drag (Exhaustion)**: higher = more exhaustion (worse).
For **Commitment Drift (Disengagement)**: higher = more disengaged (worse).

**Scoring guide:**
- Low  ≈ 2.0–2.8
- Mid  ≈ 3.0–3.8
- High ≈ 4.2–5.2


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# ■ BU SCORE PROFILES
# Each BU gets target means per subscale dimension (1–6 scale).
# Edit these to shape the organizational story for any client.
# Keys must match BU_HEADCOUNTS keys exactly.
# ═════════════════════════════════════════════════════════════════════════════

# Dimensions in order:
# Demands  : D1  D2  D3  D4  D5
# Resources: R1  R2  R3  R4  R5  R6
# 4R       : REC RES REV REF
# Outcomes : EXH DIS

BU_PROFILES = {

    # ── LEADERSHIP ────────────────────────────────────────────────────────────
    # High D1 (decision density) and D5 (boundary permeability); mid D3.
    # High R2 (rewards/recognition), low R6 (leadership reliability — they ARE leadership).
    # 4R: High Recognize, Low Respond, Mid Resolve, Low Refine.
    # High Execution Drag (exhaustion); Low Commitment Drift (still engaged).
    "Leadership": {
        "D1": 4.8, "D2": 3.2, "D3": 3.5, "D4": 2.5, "D5": 4.6,
        "R1": 2.8, "R2": 4.5, "R3": 3.8, "R4": 4.0, "R5": 3.5, "R6": 2.2,
        "REC": 4.8, "RES": 2.2, "REV": 3.4, "REF": 2.0,
        "EXH": 4.6, "DIS": 2.1,
    },

    # ── SALES ─────────────────────────────────────────────────────────────────
    # High D2 (workload compression), D3 (interpersonal complexity), D5 (boundary).
    # Low R1 (feedback clarity — unclear quota guidance) and R6 (leadership reliability).
    # Rest of R is high — compensation and autonomy strong.
    # 4R: Overall high, med Recognize, high Respond and Resolve, med Refine.
    # High Execution Drag; Low Commitment Drift.
    "Sales": {
        "D1": 3.4, "D2": 4.8, "D3": 4.6, "D4": 2.2, "D5": 4.5,
        "R1": 2.4, "R2": 4.6, "R3": 4.4, "R4": 3.8, "R5": 3.2, "R6": 2.6,
        "REC": 3.5, "RES": 4.6, "REV": 4.4, "REF": 3.6,
        "EXH": 4.7, "DIS": 2.2,
    },

    # ── OPERATIONS ────────────────────────────────────────────────────────────
    # High D1 (decision density), D3 (interpersonal complexity), D4 (environmental).
    # Low D2/D5 (workload compression and boundary are manageable).
    # Low R1 and R6.
    # High Execution Drag; Medium Commitment Drift.
    "Operations": {
        "D1": 4.5, "D2": 2.8, "D3": 4.4, "D4": 4.2, "D5": 2.6,
        "R1": 2.5, "R2": 3.0, "R3": 3.2, "R4": 3.0, "R5": 3.4, "R6": 2.6,
        "REC": 3.0, "RES": 3.2, "REV": 3.4, "REF": 2.8,
        "EXH": 4.6, "DIS": 3.5,
    },

    # ── CUSTOMER SUCCESS ──────────────────────────────────────────────────────
    # Very high D3 (interpersonal complexity with clients) and D2 (workload).
    # Resources moderate — reasonably supported but stretched.
    # 4R: Med Recognize, high Respond (trained to handle problems), med Resolve, low Refine.
    # Very high Execution Drag; rising Commitment Drift — churn risk.
    "Customer_Success": {
        "D1": 3.6, "D2": 4.5, "D3": 5.0, "D4": 2.4, "D5": 3.8,
        "R1": 3.2, "R2": 3.4, "R3": 3.0, "R4": 3.2, "R5": 3.6, "R6": 3.0,
        "REC": 3.4, "RES": 4.2, "REV": 3.2, "REF": 2.6,
        "EXH": 5.0, "DIS": 3.8,
    },

    # ── PRODUCT / SOFTWARE ────────────────────────────────────────────────────
    # High D1 (cognitive complexity). Moderate everything else.
    # Moderate resources — product team has some autonomy but caught between two platforms.
    # 4R: Low Recognize, high Respond (engineers fix things fast), low Resolve, low Refine.
    # High Execution Drag from context-switching; moderate Commitment Drift.
    "Product_Software": {
        "D1": 4.8, "D2": 3.8, "D3": 2.8, "D4": 2.6, "D5": 3.4,
        "R1": 3.4, "R2": 3.6, "R3": 4.2, "R4": 3.0, "R5": 3.2, "R6": 3.2,
        "REC": 2.8, "RES": 4.4, "REV": 2.6, "REF": 2.4,
        "EXH": 4.4, "DIS": 3.4,
    },

    # ── MARKETING ─────────────────────────────────────────────────────────────
    # Moderate D overall — lean team but focused work.
    # Reasonable resources — more autonomy than most.
    # 4R: Good Recognize and Refine (campaign learning loops); lower Respond and Resolve.
    # Moderate Execution Drag; Low Commitment Drift — engaged team.
    "Marketing": {
        "D1": 3.4, "D2": 3.6, "D3": 3.0, "D4": 2.4, "D5": 3.2,
        "R1": 3.6, "R2": 3.8, "R3": 4.0, "R4": 3.6, "R5": 3.4, "R6": 3.6,
        "REC": 4.0, "RES": 3.2, "REV": 3.0, "REF": 4.0,
        "EXH": 3.4, "DIS": 2.4,
    },

    # ── IT ────────────────────────────────────────────────────────────────────
    # High D1 (cognitive load) and D4 (environmental interference — infra chaos).
    # Very low R6 (leadership reliability — IT is often undervalued in services cos).
    # 4R: Low Recognize, high Respond (firefighting), low Resolve, very low Refine.
    # Very high Execution Drag; moderate-high Commitment Drift — flight risk.
    "IT": {
        "D1": 4.8, "D2": 3.8, "D3": 2.8, "D4": 4.6, "D5": 3.6,
        "R1": 2.8, "R2": 2.8, "R3": 3.4, "R4": 2.6, "R5": 3.0, "R6": 2.2,
        "REC": 2.4, "RES": 4.6, "REV": 2.6, "REF": 1.8,
        "EXH": 4.8, "DIS": 4.0,
    },

    # ── FINANCE ───────────────────────────────────────────────────────────────
    # Moderate-high D1 (analytical cognitive load); high D5 (month-end crunch).
    # Good resource profile — PE-focused, clear metrics, reasonable security.
    # 4R: High Recognize (FP&A sees the whole picture), moderate across rest.
    # Moderate Execution Drag; Low Commitment Drift.
    "Finance": {
        "D1": 4.0, "D2": 3.4, "D3": 2.6, "D4": 2.4, "D5": 4.4,
        "R1": 4.0, "R2": 3.8, "R3": 3.8, "R4": 3.4, "R5": 4.0, "R6": 3.8,
        "REC": 4.4, "RES": 3.6, "REV": 3.8, "REF": 3.4,
        "EXH": 3.6, "DIS": 2.4,
    },

    # ── HR ────────────────────────────────────────────────────────────────────
    # High D1 and D3 (they handle everyone else's interpersonal complexity).
    # Irony: very low R2 (rewards/recognition) and R6 for their own team.
    # 4R: High Recognize (they see org issues), low Respond (no headcount to act),
    #     low Resolve, low Refine — classic stretched HR profile.
    # High Execution Drag; moderate-high Commitment Drift.
    "HR": {
        "D1": 4.4, "D2": 3.6, "D3": 4.8, "D4": 2.4, "D5": 3.8,
        "R1": 2.6, "R2": 2.2, "R3": 3.0, "R4": 2.8, "R5": 3.0, "R6": 2.4,
        "REC": 4.6, "RES": 2.4, "REV": 2.6, "REF": 2.2,
        "EXH": 4.6, "DIS": 3.8,
    },
}

print("BU profiles loaded:")
for bu, p in BU_PROFILES.items():
    print(f"  {bu:<22}  EXH={p['EXH']:.1f}  DIS={p['DIS']:.1f}  "
          f"D_avg={np.mean([p['D1'],p['D2'],p['D3'],p['D4'],p['D5']]):.1f}  "
          f"R_avg={np.mean([p['R1'],p['R2'],p['R3'],p['R4'],p['R5'],p['R6']]):.1f}")


BU profiles loaded:
  Leadership              EXH=4.6  DIS=2.1  D_avg=3.7  R_avg=3.5
  Sales                   EXH=4.7  DIS=2.2  D_avg=3.9  R_avg=3.5
  Operations              EXH=4.6  DIS=3.5  D_avg=3.7  R_avg=2.9
  Customer_Success        EXH=5.0  DIS=3.8  D_avg=3.9  R_avg=3.2
  Product_Software        EXH=4.4  DIS=3.4  D_avg=3.5  R_avg=3.4
  Marketing               EXH=3.4  DIS=2.4  D_avg=3.1  R_avg=3.7
  IT                      EXH=4.8  DIS=4.0  D_avg=3.9  R_avg=2.8
  Finance                 EXH=3.6  DIS=2.4  D_avg=3.4  R_avg=3.8
  HR                      EXH=4.6  DIS=3.8  D_avg=3.8  R_avg=2.7


## 6. Business Unit Generator
Core function that produces the respondent-level item data for a single BU.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ITEM GROUPS — maps dimension keys to Q-numbers
# ─────────────────────────────────────────────────────────────────────────────
ITEM_GROUPS = {
    "D1":  ["D1Q1",  "D1Q2",  "D1Q3"],
    "D2":  ["D2Q1",  "D2Q2",  "D2Q3"],
    "D3":  ["D3Q1",  "D3Q2",  "D3Q3"],
    "D4":  ["D4Q1",  "D4Q2",  "D4Q3"],
    "D5":  ["D5Q1",  "D5Q2",  "D5Q3"],
    "R1":  ["R1Q1",  "R1Q2",  "R1Q3"],
    "R2":  ["R2Q1",  "R2Q2",  "R2Q3"],
    "R3":  ["R3Q1",  "R3Q2",  "R3Q3"],
    "R4":  ["R4Q1",  "R4Q2",  "R4Q3"],
    "R5":  ["R5Q1",  "R5Q2",  "R5Q3"],
    "R6":  ["R6Q1",  "R6Q2",  "R6Q3"],
    "REC": ["RECQ1", "RECQ2", "RECQ3"],
    "RES": ["RESQ1", "RESQ2", "RESQ3"],
    "REV": ["REVQ1", "REVQ2", "REVQ3"],
    "REF": ["REFQ1", "REFQ2", "REFQ3"],
    "EXH": ["EXHQ1", "EXHQ2", "EXHQ3", "EXHQ4", "EXHQ5", "EXHQ6", "EXHQ7", "EXHQ8"],
    "DIS": ["DISQ1", "DISQ2", "DISQ3", "DISQ4", "DISQ5", "DISQ6", "DISQ7", "DISQ8"],
}
ALL_ITEM_COLS = [col for items in ITEM_GROUPS.values() for col in items]


def generate_bu(
    bu_name: str,
    total_employees: int,
    profile: dict,
    calibrated_weights: dict,
    job_levels: list,
    rng,
    loading: float = 0.75,
    error_sd: float = 0.60,
    noise: float = 0.85,
) -> pd.DataFrame:
    """
    Generate a synthetic TIA respondent dataset for one business unit.

    Parameters
    ----------
    bu_name           : business unit label (string)
    total_employees   : total headcount for this BU (before participation filter)
    profile           : BU score profile dict — target means per subscale (1–6)
    level_weights     : participation probability by job level (used for level
                        composition, not participation count)
    job_levels        : list of valid job level values (e.g. [1,2,3,4,5,6])
    participation_rate: org-wide participation rate applied uniformly per BU
    rng               : numpy Generator (shared, pass same object for reproducibility)
    loading           : item factor loading applied to all items
    error_sd          : item-level error standard deviation
    noise             : latent construct noise standard deviation

    Returns
    -------
    pd.DataFrame with one row per respondent, columns:
        employee_id, business_unit, job_level, Q1–Q61

    Notes
    -----
    Participation is computed as int(round(total_employees * participation_rate))
    to give a predictable headcount. Job level is assigned proportionally
    using level_weights so that senior levels are underrepresented among
    respondents, matching real survey dynamics.
    """

    # ── 1. Determine respondent count and job level distribution ──────────────
    # Assign a job level to every employee in the BU
    raw_weights = np.array([calibrated_weights[lv] for lv in job_levels])
    level_probs = raw_weights / raw_weights.sum()
    all_levels = rng.choice(job_levels, size=total_employees, p=level_probs)

    # Each employee participates based on their level's calibrated probability
    participate_probs = np.array([calibrated_weights[lv] for lv in all_levels])
    participated = rng.random(total_employees) < participate_probs

    levels_r = all_levels[participated]
    n = max(1, len(levels_r))
    z_level = z_score(levels_r)

    # ── 2. Build latent constructs from profile targets ───────────────────────
    # Map 1–6 target to a latent shift: target - centre_of_scale / loading
    def latent(dim_key, level_eff=0.0):
        target = profile[dim_key]
        shift = (target - 3.5) / loading
        return build_latent(n, mean_shift=shift, level_effect=level_eff,
                            z_level=z_level, rng=rng, noise=noise)

    # Demands — higher job level = slightly more demand exposure
    d1_lat = latent("D1", level_eff= 0.05)
    d2_lat = latent("D2", level_eff= 0.08)
    d3_lat = latent("D3", level_eff= 0.03)
    d4_lat = latent("D4", level_eff=-0.05)
    d5_lat = latent("D5", level_eff= 0.06)

    # Resources — higher job level = slightly more access to resources
    r1_lat = latent("R1", level_eff=-0.05)
    r2_lat = latent("R2", level_eff= 0.08)
    r3_lat = latent("R3", level_eff= 0.10)
    r4_lat = latent("R4", level_eff= 0.05)
    r5_lat = latent("R5", level_eff=-0.05)
    r6_lat = latent("R6", level_eff= 0.05)

    # 4R — senior levels score slightly higher on capabilities
    rec_lat = latent("REC", level_eff=-0.08)
    res_lat = latent("RES", level_eff= 0.05)
    rev_lat = latent("REV", level_eff= 0.06)
    ref_lat = latent("REF", level_eff=-0.05)

    # Outcomes — exhaustion slightly higher at mid-levels (middle management squeeze)
    exh_lat = latent("EXH", level_eff= 0.05)
    dis_lat = latent("DIS", level_eff=-0.03)

    # ── 3. Generate items ─────────────────────────────────────────────────────
    latents = {
        "D1": d1_lat, "D2": d2_lat, "D3": d3_lat, "D4": d4_lat, "D5": d5_lat,
        "R1": r1_lat, "R2": r2_lat, "R3": r3_lat, "R4": r4_lat,
        "R5": r5_lat, "R6": r6_lat,
        "REC": rec_lat, "RES": res_lat, "REV": rev_lat, "REF": ref_lat,
        "EXH": exh_lat, "DIS": dis_lat,
    }

    item_data = {}
    for dim, items in ITEM_GROUPS.items():
        lat = latents[dim]
        # Use neutral intercept=3.5 — the latent shift already encodes
        # the target mean; adding the target again would double-count.
        item_data.update(
            make_items_for_dim(lat, items, REVERSE_MAP,
                               intercept=3.5, loading=loading,
                               error_sd=error_sd, rng=rng)
        )

    # ── 4. Assemble dataframe ─────────────────────────────────────────────────
    df = pd.DataFrame(item_data)
    df.insert(0, "job_level",      levels_r)
    df.insert(0, "business_unit",  bu_name)
    df.insert(0, "employee_id",    [f"{bu_name[:3].upper()}_{i:04d}" for i in range(n)])

    return df


print("BU generator function defined.")


BU generator function defined.


## 7. Generate Each Business Unit
Run the generator for each BU and inspect summary statistics.


In [ ]:
bu_dataframes = {}

for bu_name, headcount in BU_HEADCOUNTS.items():
    profile = BU_PROFILES[bu_name]
    df_bu = generate_bu(
        bu_name          = bu_name,
        total_employees  = headcount,
        profile          = profile,
        calibrated_weights = CALIBRATED_WEIGHTS,
        job_levels       = JOB_LEVELS,
        rng              = rng,
        loading          = 0.75,
        error_sd         = 0.60,
        noise            = 0.85,
    )
    bu_dataframes[bu_name] = df_bu
    print(f"  {bu_name:<22}  headcount={headcount:>3}  respondents={len(df_bu):>3}")

total_r = sum(len(d) for d in bu_dataframes.values())
print(f"\nTotal respondents: {total_r}  (target ~{int(TOTAL_EMPLOYEES * PARTICIPATION_RATE)})")


  Sales                   headcount=146  respondents=114
  Customer_Success        headcount= 81  respondents= 64
  Operations              headcount= 74  respondents= 62
  Product_Software        headcount= 31  respondents= 19
  Marketing               headcount= 28  respondents= 20
  IT                      headcount= 19  respondents= 14
  Finance                 headcount=  9  respondents=  6
  HR                      headcount=  7  respondents=  7
  Leadership              headcount=  6  respondents=  6

Total respondents: 312  (target ~316)


## 8. Merge to Organization-Level Dataset
Combine all BU dataframes, reset IDs, and add a global respondent ID.


In [ ]:
# ── Merge all BUs ─────────────────────────────────────────────────────────────
df_org = pd.concat(list(bu_dataframes.values()), ignore_index=True)

# Assign a globally unique respondent ID
df_org.insert(0, "respondent_id",
              [f"NXB_{i:04d}" for i in range(1, len(df_org) + 1)])

# Preserve original BU-scoped employee_id as bu_employee_id
df_org.rename(columns={"employee_id": "bu_employee_id"}, inplace=True)

print(f"Merged dataset shape: {df_org.shape}")
print(f"  Rows (respondents) : {len(df_org)}")
print(f"  Columns            : {df_org.shape[1]}")
print(f"\nBU breakdown:")
print(df_org.groupby("business_unit").size().rename("respondents").to_string())
print(f"\nJob level distribution:")
print(df_org.groupby("job_level").size().rename("count").to_string())


Merged dataset shape: (312, 65)
  Rows (respondents) : 312
  Columns            : 65

BU breakdown:
business_unit
Customer_Success     64
Finance               6
HR                    7
IT                   14
Leadership            6
Marketing            20
Operations           62
Product_Software     19
Sales               114

Job level distribution:
job_level
1    87
2    79
3    56
4    30
5    35
6    25


## 9. Computed Subscale Means (QA Only)
These columns are for quality-assurance inspection. They reflect scored-direction
means (reversals applied) so you can verify the profile shapes are correct.
Column names are prefixed with `qa_` to distinguish from raw item data.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# QA SUBSCALE MEANS
# Apply reversal in-memory for verification. Raw items are NOT modified.
# ─────────────────────────────────────────────────────────────────────────────
def scored_mean(df, items, reverse_map):
    cols = []
    for k in items:
        col = df[k].copy().astype(int)
        if reverse_map[k]:
            col = pd.Series(reverse_6(col), index=df.index)
        cols.append(col)
    return pd.concat(cols, axis=1).mean(axis=1)


for dim, items in ITEM_GROUPS.items():
    col_name = f"qa_{TERM.get(dim, dim.lower())}"
    df_org[col_name] = scored_mean(df_org, items, REVERSE_MAP)

qa_cols = [c for c in df_org.columns if c.startswith("qa_")]

print("QA subscale means added:")
print(df_org[["business_unit"] + qa_cols].groupby("business_unit").mean().round(2).to_string())


QA subscale means added:
                  qa_decision_density  qa_workload_compression  qa_interpersonal_complexity  qa_environmental_interference  qa_boundary_permeability  qa_feedback_clarity  qa_rewards_recognition  qa_decision_latitude  qa_influence_inclusion  qa_job_security  qa_leadership_reliability  qa_recognize  qa_respond  qa_resolve  qa_refine  qa_execution_drag  qa_commitment_drift
business_unit                                                                                                                                                                                                                                                                                                                                                                       
Customer_Success                 3.41                     4.46                         4.92                           2.53                      3.93                 3.21                    3.35                  2.98              

## 10. Profile Validation
Compare generated subscale means against target profile values.


In [ ]:
print("=" * 72)
print(f"{'BU':<22} {'Dim':<8} {'Target':>8} {'Generated':>10} {'Delta':>8}")
print("=" * 72)

dim_to_qa = {
    "D1": "qa_decision_density",
    "D2": "qa_workload_compression",
    "D3": "qa_interpersonal_complexity",
    "D4": "qa_environmental_interference",
    "D5": "qa_boundary_permeability",
    "R1": "qa_feedback_clarity",
    "R2": "qa_rewards_recognition",
    "R3": "qa_decision_latitude",
    "R4": "qa_influence_inclusion",
    "R5": "qa_job_security",
    "R6": "qa_leadership_reliability",
    "REC": "qa_recognize",
    "RES": "qa_respond",
    "REV": "qa_resolve",
    "REF": "qa_refine",
    "EXH": "qa_execution_drag",
    "DIS": "qa_commitment_drift",
}

for bu in BU_HEADCOUNTS.keys():
    sub = df_org[df_org["business_unit"] == bu]
    profile = BU_PROFILES[bu]
    for dim, qa_col in dim_to_qa.items():
        target = profile[dim]
        actual = sub[qa_col].mean()
        delta  = actual - target
        flag   = " ◄" if abs(delta) > 0.4 else ""
        print(f"{bu:<22} {dim:<8} {target:>8.2f} {actual:>10.2f} {delta:>+8.2f}{flag}")
    print()


BU                     Dim        Target  Generated    Delta
Sales                  D1           3.40       3.35    -0.05
Sales                  D2           4.80       4.82    +0.02
Sales                  D3           4.60       4.64    +0.04
Sales                  D4           2.20       2.23    +0.03
Sales                  D5           4.50       4.48    -0.02
Sales                  R1           2.40       2.55    +0.15
Sales                  R2           4.60       4.50    -0.10
Sales                  R3           4.40       4.37    -0.03
Sales                  R4           3.80       3.96    +0.16
Sales                  R5           3.20       3.21    +0.01
Sales                  R6           2.60       2.52    -0.08
Sales                  REC          3.50       3.47    -0.03
Sales                  RES          4.60       4.51    -0.09
Sales                  REV          4.40       4.35    -0.05
Sales                  REF          3.60       3.57    -0.03
Sales                  E

## 11. Finalize & Export

The final exported dataset contains:
- `respondent_id`, `bu_employee_id`, `business_unit`, `job_level`
- `Q1` through `Q61` — raw survey responses (reverse-scored items stored as-is)
- `qa_*` columns — scored-direction means for inspection (can be dropped before modeling)


In [ ]:
# ── Column ordering ───────────────────────────────────────────────────────────
id_cols  = ["respondent_id", "bu_employee_id", "business_unit", "job_level"]
q_cols   = ALL_ITEM_COLS
qa_cols  = [c for c in df_org.columns if c.startswith("qa_")]

df_final = df_org[id_cols + q_cols + qa_cols].copy()

# ── Final QA summary ──────────────────────────────────────────────────────────
print(f"Final dataset: {df_final.shape[0]} rows × {df_final.shape[1]} columns")
print(f"  ID columns  : {len(id_cols)}")
print(f"  Item columns: {len(q_cols)}")
print(f"  QA columns  : {len(qa_cols)}")
print(f"\nLikert range check (Q1–Q61): min={df_final[q_cols].min().min()}, max={df_final[q_cols].max().max()}")
print(f"\nMissing values: {df_final.isnull().sum().sum()}")
print(f"\nSample rows:")
print(df_final.head(3).to_string())


Final dataset: 312 rows × 82 columns
  ID columns  : 4
  Item columns: 61
  QA columns  : 17

Likert range check (Q1–Q61): min=1, max=6

Missing values: 0

Sample rows:
  respondent_id bu_employee_id business_unit  job_level  D1Q1  D1Q2  D1Q3  D2Q1  D2Q2  D2Q3  D3Q1  D3Q2  D3Q3  D4Q1  D4Q2  D4Q3  D5Q1  D5Q2  D5Q3  R1Q1  R1Q2  R1Q3  R2Q1  R2Q2  R2Q3  R3Q1  R3Q2  R3Q3  R4Q1  R4Q2  R4Q3  R5Q1  R5Q2  R5Q3  R6Q1  R6Q2  R6Q3  RECQ1  RECQ2  RECQ3  RESQ1  RESQ2  RESQ3  REVQ1  REVQ2  REVQ3  REFQ1  REFQ2  REFQ3  EXHQ1  EXHQ2  EXHQ3  EXHQ4  EXHQ5  EXHQ6  EXHQ7  EXHQ8  DISQ1  DISQ2  DISQ3  DISQ4  DISQ5  DISQ6  DISQ7  DISQ8  qa_decision_density  qa_workload_compression  qa_interpersonal_complexity  qa_environmental_interference  qa_boundary_permeability  qa_feedback_clarity  qa_rewards_recognition  qa_decision_latitude  qa_influence_inclusion  qa_job_security  qa_leadership_reliability  qa_recognize  qa_respond  qa_resolve  qa_refine  qa_execution_drag  qa_commitment_drift
0      NXB_0001       SAL

## 12. Save Dataset
Exports the dataset in two versions:
- **Full** (with qa_ columns) — for internal validation and presentation
- **Raw only** (Q1–Q61 + demographics) — for modeling pipeline input


In [ ]:
OUTPUT_PREFIX = "NexBridge_TIA_Synthetic"

# Full dataset (with QA columns)
full_path = f"{OUTPUT_PREFIX}_FULL.csv"
df_final.to_csv(full_path, index=False)
print(f"Saved (full):    {full_path}  ({df_final.shape[0]} rows)")

# Raw-only dataset (for modeling)
df_raw = df_final[id_cols + q_cols].copy()
raw_path = f"{OUTPUT_PREFIX}_RAW.csv"
df_raw.to_csv(raw_path, index=False)
print(f"Saved (raw):     {raw_path}  ({df_raw.shape[0]} rows)")

# ── Reverse scoring manifest ──────────────────────────────────────────────────
# Export a reference CSV mapping every item to its dimension and reverse flag
manifest_rows = []
for dim, items in ITEM_GROUPS.items():
    corp_name = TERM.get(dim, dim)
    for position, col in enumerate(items, 1):
        manifest_rows.append({
            "column":          col,
            "dimension_code":  dim,
            "dimension_label": corp_name,
            "item_position":   position,
            "reverse_scored":  REVERSE_MAP[col],
        })
df_manifest = pd.DataFrame(manifest_rows)
manifest_path = f"{OUTPUT_PREFIX}_ItemManifest.csv"
df_manifest.to_csv(manifest_path, index=False)
print(f"Saved (manifest): {manifest_path}  ({len(df_manifest)} items)")


Saved (full):    NexBridge_TIA_Synthetic_FULL.csv  (312 rows)
Saved (raw):     NexBridge_TIA_Synthetic_RAW.csv  (312 rows)
Saved (manifest): NexBridge_TIA_Synthetic_ItemManifest.csv  (61 items)


## 13. Organization-Level Summary Report
Printable summary of key TIA metrics across all BUs — suitable for executive review.


In [ ]:
print("=" * 72)
print(f"  {COMPANY_NAME.upper()} — TIA ORGANIZATIONAL SUMMARY")
print(f"  {INDUSTRY}")
print(f"  Revenue: {REVENUE_ARR} ARR  |  Acquisition: {ACQUISITION}")
print("=" * 72)
print(f"  Respondents: {len(df_final)} of {TOTAL_EMPLOYEES} employees "
      f"({len(df_final)/TOTAL_EMPLOYEES:.1%} participation)")

summary_dims = [
    ("DEMANDS",     ["qa_decision_density","qa_workload_compression",
                     "qa_interpersonal_complexity","qa_environmental_interference",
                     "qa_boundary_permeability"]),
    ("RESOURCES",   ["qa_feedback_clarity","qa_rewards_recognition",
                     "qa_decision_latitude","qa_influence_inclusion",
                     "qa_job_security","qa_leadership_reliability"]),
    ("4R CAPABILITY",["qa_recognize","qa_respond","qa_resolve","qa_refine"]),
    ("OUTCOMES",    ["qa_execution_drag","qa_commitment_drift"]),
]

for section, cols in summary_dims:
    print(f"\n  {section}")
    print(f"  {'-'*50}")
    for col in cols:
        label = col.replace("qa_", "").replace("_", " ").title()
        org_mean = df_final[col].mean()
        bu_means = df_final.groupby("business_unit")[col].mean()
        high_bu  = bu_means.idxmax()
        low_bu   = bu_means.idxmin()
        print(f"    {label:<30} Org={org_mean:.2f}  "
              f"High={high_bu}({bu_means[high_bu]:.2f})  "
              f"Low={low_bu}({bu_means[low_bu]:.2f})")


  NEXBRIDGE — TIA ORGANIZATIONAL SUMMARY
  Tech-Enabled Workforce Compliance & Benefits Administration
  Revenue: ~$85–110M ARR  |  Acquisition: 18 months ago
  Respondents: 312 of 401 employees (77.8% participation)

  DEMANDS
  --------------------------------------------------
    Decision Density               Org=3.78  High=HR(5.05)  Low=Marketing(2.90)
    Workload Compression           Org=4.10  High=Sales(4.82)  Low=Operations(2.86)
    Interpersonal Complexity       Org=4.27  High=Customer_Success(4.92)  Low=Product_Software(2.51)
    Environmental Interference     Org=2.83  High=IT(4.71)  Low=Marketing(2.15)
    Boundary Permeability          Org=3.78  High=Finance(4.50)  Low=Operations(2.60)

  RESOURCES
  --------------------------------------------------
    Feedback Clarity               Org=2.85  High=Finance(4.00)  Low=HR(2.19)
    Rewards Recognition            Org=3.72  High=Sales(4.50)  Low=HR(2.10)
    Decision Latitude              Org=3.73  High=Sales(4.37)  Low=H